In [ ]:
import os
import pandas as pd

# Go from notebooks/ → project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print("Project root:", PROJECT_ROOT)

SONG_LEVEL_DIR = os.path.join(
    PROJECT_ROOT,
    "data/raw/DEAM_Annotations/annotations/annotations averaged per song/song_level"
)

print("Song level dir exists:", os.path.exists(SONG_LEVEL_DIR))





In [ ]:
FILES = [
    "static_annotations_averaged_songs_1_2000.csv",
    "static_annotations_averaged_songs_2000_2058.csv"
]

dfs = []

for file in FILES:
    path = os.path.join(SONG_LEVEL_DIR, file)
    print("Loading:", path)
    dfs.append(pd.read_csv(path))

annotations_df = pd.concat(dfs, ignore_index=True)

print("Combined shape:", annotations_df.shape)
display(annotations_df.head())


In [ ]:
for col in annotations_df.columns:
    print(col)


In [ ]:
# Remove leading/trailing spaces from column names
annotations_df.columns = annotations_df.columns.str.strip()

# Verify
annotations_df.columns


In [ ]:
labels_df = annotations_df[
    ["song_id", "valence_mean", "arousal_mean"]
].copy()

labels_df.head()


In [ ]:
import os

LABELS_DIR = "../data/labels"
os.makedirs(LABELS_DIR, exist_ok=True)

labels_df.to_csv(
    os.path.join(LABELS_DIR, "deam_song_level_labels.csv"),
    index=False
)


In [ ]:
!pip install librosa soundfile numpy tqdm


In [ ]:
import os
import numpy as np
import librosa
from tqdm import tqdm


In [ ]:
AUDIO_DIR = "../data/raw/DEAM_audio/MEMD_audio"
OUTPUT_DIR = "../data/processed/mel_spectrograms"

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
def extract_mel_spectrogram(
    file_path,
    sr=22050,
    duration=30,
    n_mels=128
):
    try:
        # Load audio (force fixed duration)
        y, sr = librosa.load(
            file_path,
            sr=sr,
            duration=duration
        )

        # Pad if audio is shorter
        if len(y) < sr * duration:
            pad_width = sr * duration - len(y)
            y = np.pad(y, (0, pad_width))

        # Mel spectrogram
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=n_mels
        )

        # Convert to log scale
        mel_db = librosa.power_to_db(mel, ref=np.max)

        return mel_db

    except Exception as e:
        print("Error processing:", file_path, e)
        return None


In [ ]:
audio_files = sorted(os.listdir(AUDIO_DIR))

print("Total audio files:", len(audio_files))


In [ ]:
for file in tqdm(audio_files):
    if not file.endswith(".mp3"):
        continue

    song_id = file.replace(".mp3", "")
    audio_path = os.path.join(AUDIO_DIR, file)

    mel_spec = extract_mel_spectrogram(audio_path)

    if mel_spec is not None:
        save_path = os.path.join(OUTPUT_DIR, f"{song_id}.npy")
        np.save(save_path, mel_spec)


In [ ]:
test_file = os.listdir(OUTPUT_DIR)[0]
spec = np.load(os.path.join(OUTPUT_DIR, test_file))

print("Shape:", spec.shape)
